# 8760_aleatoric: year-long scenario campaign

* conda: `h-py312-basic`
* pipeline validation: [`m_to_case_helper.ipynb`](m_to_case_helper.ipynb)
* design notes: [`work-notes/m_to_case_helper.md`](../work-notes/m_to_case_helper.md)

Drives the validated `.m` → `case.json` → `DynamicSimulation` pipeline across an
hourly profile. **Run `m_to_case_helper.ipynb` first**; if the mapping does not
validate there, nothing here is interpretable.

## What kind of uncertainty this is

**Aleatoric.** Wind availability and load are not controllable and not reducible
by better models or more data. The question is how that irreducible variability
propagates into stability margins, and how it interacts with the epistemic
(parameter) uncertainty studied elsewhere.

Commitment is **frozen** for the whole year. Only injections vary, so the device
list never changes, `element_id` numbering stays constant, and each hour only
patches bus `init`, Genrou `p0/q0`, and `LoadZIP`.

## Read this before interpreting any result

ACTIVSg200 has **no inverter-based generation**. Verified in the source data:
`ACTIVSg200_dynamics.dyr` contains only `GENROU`, `SEXS` and `TGOV1`, 49 of each,
and all six wind-labelled units (buses 65, 104, 105, 114, 115, 147) carry a full
synchronous machine with real inertia (bus 65 `H = 6.29`, bus 114 `H = 8.92`).
The `genfuel` label is cosmetic.

Three consequences, all of which bound what this campaign can claim:

1. **Inertia is constant across every hour.** `H·MVA` is a machine parameter,
   independent of dispatch, and commitment is frozen. In reality high wind
   displaces synchronous units through unit commitment and inertia falls. That
   pathway is structurally absent here, so this campaign measures injection
   variability *at fixed inertia*, not wind penetration.
2. **Wind units provide governor response they should not.** They have `TGOV1`
   and, measured from the seed case, hold **47.6 % of online governor headroom on
   5.3 % of generation** (coal 38.7 %, gas 13.7 %, nuclear 0 %). Any
   `redistribute="headroom"` call hands roughly half the deficit to
   non-dispatchable machines. See the caveat in the commitment cell.
3. **Fault response is synchronous.** These buses contribute short-circuit
   current and a swing signature no Type 3/4 converter would produce, which
   directly affects the voltage and angle QoIs.

GridKit does implement `Regca` and the JSON parser dispatches it, but `REECA` and
`REGCB` are README-only stubs, so there is no electrical control to drive it yet.

## Sections

1. Setup
2. Base case and frozen commitment
3. Hourly profiles
4. One hour, end to end
5. Batch driver (serial here, SLURM array for the real run)
6. Collect and reduce
7. Aleatoric spread of the QoIs


In [ ]:
import importlib
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

GRIDKIT = Path.home() / "gridkit"
PY_UTILS = GRIDKIT / "uq-usecase/py-utils"
if str(PY_UTILS) not in sys.path:
    sys.path.insert(0, str(PY_UTILS))

import aux_to_m
import m_to_case_utils
import perturb
import pm_utils
import solver_utils

for _m in (m_to_case_utils, aux_to_m, perturb, solver_utils, pm_utils):
    _ = importlib.reload(_m)

import m_to_case_utils as mu
import solver_utils as su

# === Case selection + tool paths ===
SPEC = mu.ILLINOIS
RUNNER = GRIDKIT / "build/application/PhasorDynamics/DynamicSimulation"
JULIA = Path.home() / "bin/julia112"
PM_DIR = GRIDKIT / "uq-usecase/pm-solver"
PM_JL = PM_DIR / "pm_solve.jl"

# === Campaign output root ===
# One subdirectory per hour. At 8760 hours this gets large; put it on
# scratch, not home, for the real run.
WORK = GRIDKIT / "uq-usecase/notebooks/test-runs/8760_aleatoric"
WORK.mkdir(parents=True, exist_ok=True)

_paths = {
    "case.json (shipped)": SPEC.case_json_path,
    "solver.json (shipped)": SPEC.solver_json_path,
    ".aux": SPEC.aux,
    ".dyr": SPEC.dyr,
    "DynamicSimulation": RUNNER,
    "julia": JULIA,
    "pm_solve.jl": PM_JL,
}
for _label, _p in _paths.items():
    print(f"  [{'OK ' if Path(_p).exists() else 'MISSING'}]  {_label:<24} {_p}")
assert all(Path(_p).exists() for _p in _paths.values()), "fix the missing paths above"
print(f"\nWORK dir: {WORK}")

## 2. Load the base case library

A campaign is **commitment × hour**. Commitment is a pre-made choice; the hour is
the aleatoric draw.

The base `.m` files are **not built here**. They are produced and PF-solved in
section 8 of [`m_to_case_helper.ipynb`](m_to_case_helper.ipynb), which owns case
construction and validation. This notebook only consumes them, which keeps a
single definition of what a base case is and leaves an auditable artifact per
commitment (`base.m`, `base_solved.m`, `pf.log`, `meta.json`).

**Run section 8 of the helper notebook first.** The cell below fails loudly if
the library is missing.

### What the library guarantees

Each entry has already been checked there:

* the commitment is feasible (decommitted MW within eligible headroom)
* the power flow converged
* only thermal units were switched; wind is never a commitment candidate and
  never absorbs decommitted output
* the fault stayed at `element_id` 0, because `element_id` indexes only
  `BusFault` devices and ACTIVSg200 ships exactly one

### What varies here, and what does not

| | Varies | Fixed |
|---|---|---|
| per hour | bus loads, wind output | commitment, fault location, fault timing |
| per commitment | which thermal units are online, system inertia | network topology, machine parameters |

Holding the disturbance fixed is what makes hours comparable; holding commitment
fixed within a campaign is what makes the inertia attribution clean.


In [ ]:
# === Rebuild the dynamic template (cheap, keeps tasks self-contained) ===
# Must match the helper notebook's setting or the machine parameters differ.
USE_CREATOR_OVERLAY = True

SHIPPED = mu.load_case(SPEC.case_json_path)
AUX = mu.parse_aux(SPEC.aux)
DYR = mu.parse_dyr(SPEC.dyr)

_rebuilt, _ = mu.build_case_from_aux_dyr(SPEC, SHIPPED, AUX, DYR)
OVERLAY = mu.make_creator_overlay(SHIPPED, _rebuilt, tol=1e-4)
BASE_CASE, _ = mu.build_case_from_aux_dyr(
    SPEC, SHIPPED, AUX, DYR, creator_overlay=OVERLAY if USE_CREATOR_OVERLAY else None
)
print(f"USE_CREATOR_OVERLAY = {USE_CREATOR_OVERLAY}")
print(
    f"BASE_CASE vs shipped: {len(mu.diff_case_json(SHIPPED, BASE_CASE, tol=1e-6))} diffs"
)

# === Load the base case library built by m_to_case_helper section 8 ===
HELPER_WORK = GRIDKIT / "uq-usecase/notebooks/test-runs/m_to_case_helper"
BASE_DIR = HELPER_WORK / "bases"

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"no base case library at {BASE_DIR}\n"
        "Run section 8 of m_to_case_helper.ipynb first."
    )

BASES = {}
for meta in sorted(BASE_DIR.glob("*/meta.json")):
    rec = json.loads(meta.read_text())
    if rec.get("m") is None:
        print(f"  skipping {rec['name']}: base PF failed when it was built")
        continue
    rec["frames"] = aux_to_m.m_to_frames(rec["m"])
    BASES[rec["name"]] = rec

if not BASES:
    raise RuntimeError(f"{BASE_DIR} has no usable base cases")

LIBRARY = pd.DataFrame(
    {
        k: {
            "off": ",".join(v["off"]) or "-",
            "lost_mw": v["lost_mw"],
            "n_online": v.get("n_online"),
            "inertia_MWs": v.get("inertia_MWs"),
            "inertia_pct": v.get("inertia_pct"),
            "V_min_pu": v.get("V_min_pu"),
        }
        for k, v in BASES.items()
    }
).T
print(f"\n{len(BASES)} base cases from {BASE_DIR}")
print(LIBRARY.to_string())

## 3. Hourly profiles

The campaign is driven by a table with one row per hour. Two accepted shapes:

**Per-bus (preferred).** Columns `hour`, then one column per load bus and per
wind unit, in MW:

```
hour,load_1,load_3,...,wind_65_1,wind_104_1,...
0,12.4,8.1,...,41.2,60.3,...
```

**System-level with shape factors.** Columns `hour`, `load_mw`, `wind_mw`, with
the system total disaggregated across buses using the base case's existing
proportions. Cruder, but usable when only aggregate profiles exist.

`PROFILE_CSV` below points at the real file. **It does not exist yet.** Until it
does, the cell builds a clearly-labelled synthetic profile so the machinery can
be smoke-tested end to end. Nothing produced from synthetic input is a result.

### Why per-bus matters

A single system-wide scale factor moves every bus together, which is not how load
or wind actually varies and removes exactly the spatial correlation structure
that makes the aleatoric question interesting. `perturb.set_bus_load` takes a
`{bus: MW}` dict precisely so the per-bus path is the default.


In [ ]:
# === Profile source (edit this path) ===
PROFILE_CSV = GRIDKIT / "uq-usecase/data/activsg200_hourly.csv"
N_HOURS = 24  # subset for development; 8760 for the real campaign

# Load buses and wind units are the same across commitments (only thermal
# units are switched), so the reference base defines the profile targets.
REF_COMMITMENT = "c0_full"
BASE = BASES[REF_COMMITMENT]["frames"]

_bus, _gen = BASE["bus"], BASE["gen"]
LOAD_BUSES = sorted(_bus.loc[_bus["PD"] != 0, "BUS_I"].astype(int))

_gid = perturb.gen_index(BASE)
_fuel = pd.Series([str(x).lower() for x in BASE["genfuel"]], index=_gen.index)
_is_wind_on = (_fuel == "wind") & (_gen["GEN_STATUS"].astype(int) == 1)
WIND_IDS = list(_gid[_is_wind_on])
WIND_PMAX = dict(zip(WIND_IDS, _gen.loc[_is_wind_on, "PMAX"].astype(float)))

print(f"{len(LOAD_BUSES)} load buses, {len(WIND_IDS)} online wind units")
print(f"wind PMAX (MW): { {k: round(v, 1) for k, v in WIND_PMAX.items()} }")

if PROFILE_CSV.exists():
    PROFILE = pd.read_csv(PROFILE_CSV).set_index("hour")
    SYNTHETIC = False
    print(f"\nprofile: {PROFILE_CSV}  {PROFILE.shape}")
else:
    # ---- SYNTHETIC, FOR SMOKE TESTING ONLY ----
    # A diurnal load shape and an AR(1) wind series. These numbers are
    # invented. Anything computed from them tests the plumbing and is
    # not a statement about the system.
    SYNTHETIC = True
    _rng = np.random.default_rng(12345)
    _h = np.arange(N_HOURS)
    _shape = 0.85 + 0.15 * np.sin(2 * np.pi * (_h - 6) / 24)

    _cols = {}
    _pd0 = dict(zip(_bus["BUS_I"].astype(int), _bus["PD"].astype(float)))
    for b in LOAD_BUSES:
        _cols[f"load_{b}"] = _pd0[b] * _shape * _rng.normal(1.0, 0.02, N_HOURS)

    _w = np.zeros(N_HOURS)
    _w[0] = 0.4
    for i in range(1, N_HOURS):  # AR(1), invented persistence
        _w[i] = np.clip(0.85 * _w[i - 1] + 0.15 * _rng.uniform(0, 1), 0.0, 1.0)
    for gid in WIND_IDS:
        _cols[f"wind_{gid}"] = WIND_PMAX[gid] * _w

    PROFILE = pd.DataFrame(_cols, index=pd.Index(_h, name="hour"))
    print(f"\n*** PROFILE_CSV not found: {PROFILE_CSV}")
    print("*** Using a SYNTHETIC profile. Smoke test only, not a result.")

HOURS = list(PROFILE.index[:N_HOURS])
print(f"\nhours available: {len(PROFILE)}, running: {len(HOURS)}")
print(PROFILE.iloc[:3, :6].round(2).to_string())

## 4. One hour, end to end

`hour_frames` turns a profile row into perturbed frames; `run_hour` takes it
through power flow, case build and dynamics. Same six steps as `run_scenario`
in the helper notebook, with three differences that matter at scale:

* starts from a **base case's** solved frames, not the raw seed, so the
  commitment under study is already applied
* takes `base_frames` and `base_case` as arguments, so one function serves every
  commitment in the library
* writes `record.json` per hour, so a failed array task can be retried without
  rerunning the rest

The fault is identical for every hour and every commitment. Only injections
vary, which is what makes hours comparable.


In [ ]:
# === Fault and timing: identical for every hour ===
# Only injections vary across the campaign, so the disturbance is held fixed.
# ACTIVSg200 ships exactly one BusFault (bus 149), so location is not a
# variable yet; duration is, via T_ON / T_OFF.
BASE_SOLVER = json.loads(Path(SPEC.solver_json_path).read_text())
FAULT_BUS = 149
TMAX, T_ON, T_OFF = 10.0, 1.0, 1.1

_fid = su.fault_element_id(BASE_CASE, FAULT_BUS)
SOLVER_OVERRIDES = {
    "tmax": TMAX,
    "events": [
        {"time": T_ON, "type": "fault_on", "element_id": _fid},
        {"time": T_OFF, "type": "fault_off", "element_id": _fid},
    ],
}
_SOLVER = su.apply_solver_overrides(BASE_SOLVER, SOLVER_OVERRIDES)
_ = su.validate_solver(_SOLVER)
print(su.describe_solver(_SOLVER, BASE_CASE))


def hour_frames(h, profile=None, base=None):
    """Apply one profile row to the base frames. No file IO."""
    profile = PROFILE if profile is None else profile
    base = BASE if base is None else base
    row = profile.loc[h]
    f = perturb.copy_frames(base)

    pd_mw = {
        b: float(row[f"load_{b}"]) for b in LOAD_BUSES if f"load_{b}" in profile.columns
    }
    if pd_mw:
        f = perturb.set_bus_load(f, pd_mw)  # QD follows, power factor held

    pg_mw = {
        g: float(row[f"wind_{g}"]) for g in WIND_IDS if f"wind_{g}" in profile.columns
    }
    if pg_mw:
        f = perturb.set_gen_output(f, pg_mw=pg_mw)
    return f


def run_hour(h, outdir=None, base_case=None, base_frames=None, keep_mon=True):
    """One hour: perturb -> PF -> case.json -> dynamics. Returns a record."""
    base_case = BASE_CASE if base_case is None else base_case
    d = Path(outdir) if outdir else WORK / "hours" / f"h{int(h):05d}"
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)
    rec = {"hour": int(h), "dir": str(d), "synthetic": SYNTHETIC}
    t0 = time.time()

    frames = hour_frames(h, base=base_frames)
    rec.update(perturb.summarize(frames, str(h)).drop("label").to_dict())
    scen_m = d / "hour.m"
    aux_to_m.write_matpower_m(frames, scen_m, case_name=f"h{int(h)}")

    solved_m = d / "hour_solved.m"
    bus_df, pf_err, pf_rc = pm_utils.run_pm_solve_out(
        JULIA, PM_DIR, PM_JL, scen_m, solved_m
    )
    (d / "pf.log").write_text(pf_err)
    _s = re.search(r"termination_status=(\S+)", pf_err)
    rec["pf_rc"] = pf_rc
    rec["pf_status"] = _s.group(1) if _s else "?"
    if pf_rc != 0 or not solved_m.exists():
        rec["error"] = "power flow failed"
        (d / "record.json").write_text(json.dumps(rec, indent=2, default=str))
        return rec
    rec["V_min_pu"] = float(bus_df["V_pu"].min())
    rec["V_max_pu"] = float(bus_df["V_pu"].max())

    case, counts = mu.build_case_from_m(SPEC, base_case, solved_m)
    rec["n_genrou"] = counts["gen"]
    case_json = d / "case.json"
    mu.write_case(case, case_json)

    solver = su.apply_solver_overrides(BASE_SOLVER, SOLVER_OVERRIDES)
    solver["system_model_file"] = case_json.name
    su.write_solver(su.validate_solver(solver), d / "solver.json")

    r = subprocess.run(
        [str(RUNNER), "solver.json"], cwd=d, capture_output=True, text=True
    )
    (d / "sim.log").write_text(r.stdout + r.stderr)
    rec["sim_rc"] = r.returncode
    rec["sim_errors"] = sum("ERROR" in ln for ln in (r.stdout + r.stderr).splitlines())
    rec["wall_s"] = round(time.time() - t0, 1)
    mon = next(iter(d.glob("*.csv")), None)
    rec["mon_csv"] = str(mon) if mon else None
    if mon and not keep_mon:
        # ~27 MB per hour; at 8760 that is ~230 GB. Reduce then delete.
        rec["mon_csv"] = None
        mon.unlink()
    (d / "record.json").write_text(json.dumps(rec, indent=2, default=str))
    return rec


print("\nhour_frames / run_hour defined")

## 5. Batch driver

Run a few hours here to confirm the loop works, then hand the rest to SLURM.

### Cost

From the helper notebook's timings, about 9 s per scenario, almost all of it
Julia startup rather than the Ipopt solve (4 iterations typical). Serial 8760
would therefore be roughly a day of wall time, which is the wrong shape for this
problem. Two independent fixes:

1. **Array job.** Embarrassingly parallel; each hour is self-contained. This is
   the easy win and is what the cell below writes.
2. **Amortise Julia startup.** Have `pm_solve.jl` loop over a batch of `.m`
   files in one process instead of one process per file. Removes most of the per
   hour cost. Nothing else in the pipeline changes.

### Disk

`mon.csv` is about 27 MB per run at `dt_monitor` as shipped. Keeping all 8760 is
roughly 230 GB. Set `keep_mon=False` in `run_hour` to reduce to QoI scalars and
delete the raw file, or raise `dt_monitor` if the full trace is not needed.


In [ ]:
# === Serial run (development) ===
# The campaign is commitment x hour. Keep both small here; the real run
# goes through the array job in the next cell.
RUN_COMMITMENTS = ["c0_full", "c3_m450"]
RUN_HOURS = HOURS[:3]

records = []
for cname in RUN_COMMITMENTS:
    b = BASES[cname]
    if b["m"] is None:
        print(f"--- {cname}: base PF failed, skipping")
        continue
    for h in RUN_HOURS:
        print(f"--- {cname} h{h} ", end="", flush=True)
        rec = run_hour(
            h,
            outdir=WORK / "runs" / cname / f"h{int(h):05d}",
            base_frames=b["frames"],
        )
        rec["commitment"] = cname
        records.append(rec)
        print(
            f"pf={rec.get('pf_status')}  sim_rc={rec.get('sim_rc')}  {rec.get('wall_s')}s"
        )

HOURLY = pd.DataFrame(records).set_index(["commitment", "hour"])
cols = [
    "pf_status",
    "V_min_pu",
    "total_PD_MW",
    "total_PG_MW",
    "PG_minus_PD_MW",
    "n_genrou",
    "sim_rc",
    "sim_errors",
    "wall_s",
]
print()
print(HOURLY[[c for c in cols if c in HOURLY]].to_string())

_bad = HOURLY.index[
    (HOURLY["pf_status"] != "LOCALLY_SOLVED")
    | (HOURLY["sim_rc"] != 0)
    | (HOURLY["sim_errors"] > 0)
].tolist()
print(f"\nfailed runs: {_bad or 'none'}")
# n_genrou should differ between commitments and be constant within one.
print("\nGenrou count per commitment:")
print(HOURLY.groupby("commitment")["n_genrou"].agg(["min", "max"]).to_string())
if SYNTHETIC:
    print("\nNOTE: synthetic profile. This checks the plumbing, nothing more.")

In [ ]:
# === SLURM array scaffold ===
# Writes a worker script and a submit script. Nothing is submitted
# from the notebook; run the sbatch line yourself when ready.
# Each task re-imports and rebuilds BASE_CASE, which costs well under
# a second and keeps tasks independent of notebook state.
ARRAY_DIR = WORK / "array"
ARRAY_DIR.mkdir(parents=True, exist_ok=True)

PY_EXE = sys.executable
ACCOUNT = "CHANGE_ME"  # SLURM account
PARTITION = "CHANGE_ME"  # SLURM partition
MAX_CONCURRENT = 50  # throttle so PM.jl does not swamp the filesystem

_worker = f'''"""One SLURM array task = one hour. Generated by 8760_aleatoric.ipynb."""
import os
import sys
from pathlib import Path

sys.path.insert(0, "{PY_UTILS}")
NB_DIR = Path("{Path.cwd()}")

hour = int(sys.argv[1] if len(sys.argv) > 1 else os.environ["SLURM_ARRAY_TASK_ID"])

# Rebuild the same objects the notebook builds. Keep this in sync, or
# refactor the shared part into py-utils and import it from both.
raise SystemExit(
    "TODO: factor cells 2-4 into py-utils/campaign.py, import here, "
    f"then call run_hour({{hour}}). Left explicit rather than duplicating "
    "notebook code that would silently drift out of sync."
)
'''
(ARRAY_DIR / "run_hour_task.py").write_text(_worker)

_sbatch = f"""#!/bin/bash
#SBATCH --job-name=aleatoric8760
#SBATCH --account={ACCOUNT}
#SBATCH --partition={PARTITION}
#SBATCH --array=0-{len(PROFILE) - 1}%{MAX_CONCURRENT}
#SBATCH --time=00:20:00
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=4
#SBATCH --output={ARRAY_DIR}/logs/h%a.out
#SBATCH --error={ARRAY_DIR}/logs/h%a.err

set -euo pipefail
mkdir -p {ARRAY_DIR}/logs
{PY_EXE} {ARRAY_DIR}/run_hour_task.py "$SLURM_ARRAY_TASK_ID"
"""
(ARRAY_DIR / "submit.sh").write_text(_sbatch)

print(f"wrote {ARRAY_DIR}/run_hour_task.py")
print(f"wrote {ARRAY_DIR}/submit.sh")
print(f"\narray size : {len(PROFILE)} tasks, {MAX_CONCURRENT} concurrent")
print("submit with : sbatch", ARRAY_DIR / "submit.sh")
print("\nBefore submitting:")
print("  1. set ACCOUNT and PARTITION above")
print("  2. resolve the TODO in run_hour_task.py")
print("  3. move WORK to scratch if keep_mon=True")

## 6. Collect and reduce

Gathers every `record.json` under `WORK/hours/` and reduces each `mon.csv` to the
same QoI scalars the helper notebook defines. Works whether the hours came from
the serial loop or the array job, since both write the same layout.

QoIs are split observable / internal for the reason set out in the helper
notebook: a PMU sees bus voltage magnitude, angle and the frequency derived from
angle. Rotor angle and rotor speed are internal states with no field equivalent,
so they are reported separately rather than mixed in.

Bus frequency is differentiated from the unwrapped angle with a blanking window
after clearing. Differentiating the clearing step itself produces a spike that is
numerical, not physical; without blanking it dominates the ROCOF statistic.


In [ ]:
# === QoI reduction (same definition as m_to_case_helper) ===
F_BASE = float(BASE_CASE["params"]["freq_base"])
V_RECOVER_PU = 0.90  # "recovered" threshold for the voltage clock
F_BLANK_S = 0.05  # ignore this long after clearing for df/dt


def load_mon(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def qoi(df, t_on=T_ON, t_off=T_OFF, f_base=F_BASE, v_thr=V_RECOVER_PU):
    """Reduce one mon.csv to scalars. Independent of machine count."""
    t = df["t"].to_numpy()
    during = (t >= t_on) & (t <= t_off)
    post = t > t_off

    vr = df[[c for c in df.columns if c.endswith("_Vr")]].to_numpy()
    vi = df[[c for c in df.columns if c.endswith("_Vi")]].to_numpy()
    vmag = np.hypot(vr, vi)
    va = np.unwrap(np.arctan2(vi, vr), axis=0)
    va_spread = np.rad2deg(va.max(axis=1) - va.min(axis=1))

    ok = (vmag >= v_thr).all(axis=1)
    bad = np.where(post & ~ok)[0]
    if bad.size == 0:
        v_recovery_s = 0.0
    elif bad[-1] + 1 < len(t):
        v_recovery_s = float(t[bad[-1] + 1] - t_off)
    else:
        v_recovery_s = np.nan

    meas = t > (t_off + F_BLANK_S)
    busf = f_base + np.gradient(va, t, axis=0) / (2.0 * np.pi)
    rocof = np.gradient(busf, t, axis=0)

    domega = df[[c for c in df.columns if c.endswith("_omega")]].to_numpy()
    freq = f_base * (1.0 + domega)
    delta = np.rad2deg(df[[c for c in df.columns if c.endswith("_delta")]].to_numpy())
    d_spread = delta.max(axis=1) - delta.min(axis=1)

    return {
        "Vmin_fault": vmag[during].min(),
        "Vmin_post": vmag[post].min(),
        "v_recovery_s": v_recovery_s,
        "max_va_spread_deg": va_spread.max(),
        "final_va_spread_deg": va_spread[-1],
        "nadir_busf_Hz": busf[meas].min(),
        "max_rocof_Hz_s": np.abs(rocof[meas]).max(),
        "nadir_f_Hz": freq.min(),
        "max_angle_spread_deg": d_spread.max(),
        "any_nan": bool(np.isnan(domega).any() or np.isnan(vmag).any()),
        "t_end": t[-1],
    }


# === Gather every completed hour ===
_recs = []
for rj in sorted((WORK / "hours").glob("h*/record.json")):
    _recs.append(json.loads(rj.read_text()))
print(f"found {len(_recs)} hour records under {WORK / 'hours'}")

RESULTS = pd.DataFrame(_recs).set_index("hour").sort_index()
_rows = {}
for h, r in RESULTS.iterrows():
    p = r.get("mon_csv")
    if p and Path(p).exists():
        _rows[h] = qoi(load_mon(p))
QOI = pd.DataFrame(_rows).T.sort_index()
QOI.index.name = "hour"

OBS = [
    "Vmin_fault",
    "Vmin_post",
    "v_recovery_s",
    "max_va_spread_deg",
    "nadir_busf_Hz",
    "max_rocof_Hz_s",
]
INTERNAL = ["nadir_f_Hz", "max_angle_spread_deg"]

if len(QOI):
    print(f"\nreduced {len(QOI)} hours")
    print("\nOBSERVABLE")
    print(QOI[OBS].to_string())
    assert not QOI["any_nan"].any(), "a run diverged"
else:
    print("\nno mon.csv found; run some hours first (or keep_mon was False)")

## 7. Aleatoric spread of the QoIs

The output of the campaign is a **distribution** per QoI, not a single number.
What to look at:

* **Tails, not means.** The interesting hours are the worst ones. A mean nadir is
  nearly meaningless; the 1st percentile and the minimum are what matter.
* **Spread against the driver.** Plot each QoI against total load and total wind.
  If the QoI is flat in wind, this case cannot see the effect, which given the
  synchronous-machine representation of wind is the expected outcome for anything
  inertia-related.
* **Worst hours.** Keep their `mon.csv`, re-run them with a longer `tmax`, and
  inspect the raw traces before drawing conclusions.

### What this distribution does and does not represent

It is the spread induced by injection variability **at fixed commitment and fixed
inertia**. It is not the spread a real system would see across a year, because
the inertia-displacement pathway is absent (see the caveats at the top).

For the epistemic side, the useful product is conditioning: how parameter
sensitivity and identifiability change across operating points. An hour where the
QoIs barely move tells you little about parameters; an hour near a stability
boundary is where calibration data is informative.


In [ ]:
if not len(QOI):
    print("nothing to summarise; run section 5 first")
else:
    # === Distribution summary, tails emphasised ===
    _stats = QOI[OBS + INTERNAL].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    print("QoI distribution across hours")
    print(_stats.round(4).to_string())

    # === Worst hours, by the two margins that matter ===
    print("\nworst 5 by frequency nadir")
    print(QOI.nsmallest(5, "nadir_busf_Hz")[OBS].round(4).to_string())
    print("\nworst 5 by angle spread")
    print(QOI.nlargest(5, "max_va_spread_deg")[OBS].round(4).to_string())

    # === QoI against the drivers ===
    # Flat in wind means this case cannot resolve the effect, which is
    # the expected result for inertia-related QoIs here.
    _drv = RESULTS.loc[QOI.index, ["total_PD_MW", "total_PG_MW"]].copy()
    _joined = _drv.join(QOI)

    for _q in ["nadir_busf_Hz", "max_va_spread_deg", "Vmin_post"]:
        fig = px.scatter(
            _joined.reset_index(),
            x="total_PD_MW",
            y=_q,
            color="total_PG_MW",
            hover_data=["hour"],
            title=f"{_q} vs load  (colour = total generation)"
            + ("  [SYNTHETIC]" if SYNTHETIC else ""),
            labels={"total_PD_MW": "total load (MW)"},
        )
        _ = fig.update_layout(height=380)
        _ = fig.show()

    if SYNTHETIC:
        print("\n*** SYNTHETIC PROFILE: these distributions are not results. ***")